# 01 - Data Preparation

## Objective
Load PLFS 2023-24 raw data, identify correct earnings columns, create analysis dataset with readable labels, and prepare for EDA.

## Key Variables
- **Gender**: b4q5_perv1 (1=Male, 2=Female, 3=Other)
- **Age**: b4q6_perv1
- **Marital Status**: b4q7_perv1
- **Education Level**: b4q8_perv1
- **Formal Education Years**: b4q10_perv1
- **Employment Status**: b5pt1q3_perv1
- **Industry (NIC)**: b5pt1q5_perv1
- **Occupation (NCO)**: b5pt1q6_perv1
- **Earnings (Self-Employed)**: b6q9_perv1
- **Earnings (Regular/Salaried)**: b6q10_perv1

In [ ]:
import pandas as pd
import numpy as np
import json

# Load raw data
df = pd.read_csv("../data/raw/perv1.csv")

print("Raw data shape:", df.shape)
print("Columns loaded successfully")

## Step 1: Create Mapping Dictionaries for Readable Labels
To make visualizations understandable (no raw codes)

In [ ]:
# Gender mapping
gender_map = {
    1: 'Male',
    2: 'Female',
    3: 'Other'
}

# Marital Status mapping (from PLFS 2023-24 Data Layout)
marital_status_map = {
    1: 'Never Married',
    2: 'Married',
    3: 'Widowed',
    4: 'Divorced',
    5: 'Separated'
}

# General Education Level mapping (from PLFS 2023-24 Data Layout)
education_level_map = {
    1: 'Not Literate',
    2: 'Literate Without Formal Schooling',
    3: 'Below Primary',
    4: 'Primary',
    5: 'Middle',
    6: 'Matric/10th',
    7: 'Higher Secondary/12th',
    8: 'Diploma/Certificate',
    10: 'Graduate',
    11: 'Post Graduate',
    12: 'Other Education',
    13: 'Technical Education'
}

# Employment Status mapping (from PLFS 2023-24 Data Layout - Status Codes)
employment_status_map = {
    11: 'Self-employed (Own Account)',
    12: 'Self-employed (Employer)',
    21: 'Regular Salaried/Wage',
    31: 'Casual Wage (Non-Agri)',
    41: 'Casual Wage (Agriculture)',
    51: 'Unpaid Family Worker',
    81: 'Domestic Worker',
    91: 'Not in Labour - Student',
    92: 'Not in Labour - Domestic',
    93: 'Not in Labour - Disabled/Ill',
    94: 'Not in Labour - Old Age',
    95: 'Not in Labour - Other',
    97: 'Not in Labour - Unspecified'
}

# NCO (Occupation) mapping - Top 15 occupations (from National Classification of Occupations)
occupation_map = {
    611.0: 'General Farming',
    522.0: 'Retail Trade Workers',
    832.0: 'Helpers & Labourers (Manufacturing)',
    612.0: 'Vegetable & Crop Farming',
    112.0: 'Corporate Managers',
    753.0: 'Machinery Mechanics',
    613.0: 'Fruit Farming',
    911.0: 'Cleaners & Helpers (Agriculture)',
    541.0: 'Personal Care Workers',
    234.0: 'Accountants',
    932.0: 'Food Preparation Assistants',
    411.0: 'Office Clerks',
    731.0: 'Welders & Flame Cutters',
    833.0: 'Drivers & Mobile Plant Operators',
    723.0: 'Electricians'
}

# NIC (Industry) mapping - Top 15 industries (from National Industrial Classification)
industry_map = {
    1.0: 'Agriculture, Forestry & Fishing',
    10.0: 'Food Product Manufacturing',
    13.0: 'Textile Manufacturing',
    18.0: 'Printing',
    20.0: 'Chemical Manufacturing',
    25.0: 'Fabricated Metal Products',
    28.0: 'Machinery Manufacturing',
    41.0: 'Electricity, Gas & Water Supply',
    45.0: 'Construction',
    47.0: 'Retail Trade',
    52.0: 'Warehouse & Support',
    56.0: 'Food & Beverage Service',
    69.0: 'Legal & Accounting',
    84.0: 'Government & Defence',
    85.0: 'Education'
}

print("✓ Mappings created for:")
print("  - Gender (3 categories)")
print("  - Marital Status (5 categories)")
print("  - Education Level (13 categories)")
print("  - Employment Status (13 categories)")
print("  - Occupation (15 top codes)")
print("  - Industry (15 top codes)")

## Step 2: Extract and Rename Features

In [ ]:
feature_mapping = {
    "gender": "b4q5_perv1",
    "age": "b4q6_perv1",
    "marital_status": "b4q7_perv1",
    "education_level": "b4q8_perv1",
    "formal_education_years": "b4q10_perv1",
    "employment_status": "b5pt1q3_perv1",
    "industry": "b5pt1q5_perv1",
    "occupation": "b5pt1q6_perv1",
    "self_employed_earnings": "b6q9_perv1",
    "regular_earnings": "b6q10_perv1"
}

analysis_df = df[list(feature_mapping.values())].copy()
analysis_df.columns = list(feature_mapping.keys())

print("✓ Analysis dataframe created")
print("  Shape:", analysis_df.shape)
print("  Columns:", list(analysis_df.columns))

## Step 3: Create Total Earnings Variable

In [ ]:
# Combine earnings from self-employed and regular activities
analysis_df["earnings"] = (
    analysis_df["self_employed_earnings"] + 
    analysis_df["regular_earnings"]
)

print("Earnings Statistics:")
print(f"  Total records: {len(analysis_df):,}")
print(f"  Earnings > 0: {(analysis_df['earnings'] > 0).sum():,}")
print(f"  Earnings = 0: {(analysis_df['earnings'] == 0).sum():,}")
print(f"  Earnings < 0: {(analysis_df['earnings'] < 0).sum():,}")

## Step 4: Filter for Positive Earners Only

In [ ]:
# Keep only people with positive earnings
earners_df = analysis_df[analysis_df["earnings"] > 0].copy()

print(f"✓ Earners dataframe created")
print(f"  Shape: {earners_df.shape}")
print(f"  Removed: {analysis_df.shape[0] - earners_df.shape[0]:,} records")

## Step 5: Remove Rows with Missing Industry or Occupation

In [ ]:
print("Before removing missing values:")
print(f"  Shape: {earners_df.shape}")
print(f"  Missing industry: {earners_df['industry'].isna().sum():,}")
print(f"  Missing occupation: {earners_df['occupation'].isna().sum():,}")

# Remove missing industry and occupation
clean_df = earners_df[
    (earners_df["industry"].notna()) & 
    (earners_df["occupation"].notna())
].copy()

print("\n✓ After removing missing values:")
print(f"  Shape: {clean_df.shape}")
print(f"  Records removed: {earners_df.shape[0] - clean_df.shape[0]:,}")

## Step 6: Summary Statistics

In [ ]:
print("="*70)
print("FINAL CLEAN DATASET SUMMARY")
print("="*70)
print(f"\nFinal dataset shape: {clean_df.shape}")
print(f"Records: {clean_df.shape[0]:,}")
print(f"Variables: {clean_df.shape[1]}")

print("\n" + "="*70)
print("EARNINGS SUMMARY")
print("="*70)
print(f"\nMean earnings: ₹{clean_df['earnings'].mean():,.2f}")
print(f"Median earnings: ₹{clean_df['earnings'].median():,.2f}")
print(f"Std Dev: ₹{clean_df['earnings'].std():,.2f}")
print(f"Min: ₹{clean_df['earnings'].min():,.2f}")
print(f"Max: ₹{clean_df['earnings'].max():,.2f}")

print("\n" + "="*70)
print("GENDER-WISE EARNINGS BREAKDOWN")
print("="*70)
gender_earnings = clean_df.groupby('gender')['earnings'].agg(['count', 'mean', 'median', 'std'])
for idx, row in gender_earnings.iterrows():
    gender_name = gender_map.get(idx, f"Code {idx}")
    print(f"\n{gender_name}:")
    print(f"  Count: {int(row['count']):,}")
    print(f"  Mean: ₹{row['mean']:,.2f}")
    print(f"  Median: ₹{row['median']:,.2f}")
    print(f"  Std Dev: ₹{row['std']:,.2f}")

print("\n" + "="*70)
print("GENDER GAP ANALYSIS")
print("="*70)
male_median = gender_earnings.loc[1, 'median']
female_median = gender_earnings.loc[2, 'median']
gap = male_median - female_median
gap_pct = (gap / male_median) * 100
print(f"\nMedian Earnings Gap (Male - Female): ₹{gap:,.2f}")
print(f"Gap as % of Male earnings: {gap_pct:.2f}%")

## Step 7: Save Clean Dataset and Mappings

In [ ]:
# Save clean dataset
clean_df.to_csv("../data/processed/clean_earners.csv", index=False)
print("✓ Saved: clean_earners.csv")

# Save mappings as JSON for reuse in EDA
mappings = {
    "gender": gender_map,
    "marital_status": marital_status_map,
    "education_level": education_level_map,
    "employment_status": employment_status_map,
    "occupation": occupation_map,
    "industry": industry_map
}

# Convert integer keys to strings for JSON compatibility
mappings_for_json = {}
for key, mapping in mappings.items():
    mappings_for_json[key] = {str(int(k) if isinstance(k, float) else k): v for k, v in mapping.items()}

with open("../data/processed/mappings.json", "w") as f:
    json.dump(mappings_for_json, f, indent=2)

print("✓ Saved: mappings.json")
print("\n" + "="*70)
print("DATA PREPARATION COMPLETE!")
print("Ready for EDA.")
print("="*70)